import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

Final Performance Table

In [ ]:
final_test_results = []

for name, model in final_models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    report = classification_report(
        y_test,
        y_pred,
        target_names=["Dropout", "Enrolled", "Graduate"],
        output_dict=True,
        zero_division=0
    )

    final_test_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (macro)": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "Recall (macro)": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "F1 Macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "Dropout F1": report["Dropout"]["f1-score"],
        "Enrolled F1": report["Enrolled"]["f1-score"],
        "Graduate F1": report["Graduate"]["f1-score"],
    })

final_perf_df = pd.DataFrame(final_test_results).round(4)

metric_cols = [
    "Accuracy", "Precision (macro)", "Recall (macro)", "F1 Macro",
    "Dropout F1", "Enrolled F1", "Graduate F1"
]

styled_final_perf_df = (
    final_perf_df.style
    .highlight_max(subset=metric_cols, color="#d9ead3")   
    .highlight_min(subset=metric_cols, color="#f4cccc")   
)

display(styled_final_perf_df)

,Model,Accuracy,Precision (macro),Recall (macro),F1 Macro,Dropout F1,Enrolled F1,Graduate F1
0,Logistic Regression (tuned),0.769500,0.709100,0.677500,0.684900,0.780000,0.413800,0.861100
1,Decision Tree,0.694900,0.635700,0.637700,0.636500,0.697300,0.412100,0.800000
2,Random Forest,0.763800,0.715000,0.674300,0.684700,0.774500,0.432400,0.847000
3,SVM RBF (tuned),0.748000,0.690200,0.667100,0.674700,0.760500,0.427000,0.836500
4,XGBoost,0.769500,0.718400,0.705900,0.711100,0.775100,0.503300,0.855000


Extreme Error Analysis - XGBoost vs. Random Forest 

In [59]:
for name in ["XGBoost", "Random Forest"]:
    model = final_models[name]
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)

    results = pd.DataFrame({
        "true": le.inverse_transform(y_test),
        "pred": le.inverse_transform(y_pred),
        "confidence": y_proba.max(axis=1),
    })
    results["correct"] = results["true"] == results["pred"]
    extreme = results[~results["correct"] & (results["confidence"] > 0.7)]

    print(f"\n=== {name} ===")
    print(f"Total errors: {(~results['correct']).sum()}")
    print(f"Extreme errors (confidence > 0.7): {len(extreme)}")
    print(extreme.groupby(["true", "pred"]).size().reset_index(name="count").sort_values("count", ascending=False))


=== XGBoost ===
Total errors: 204
Extreme errors (confidence > 0.7): 119
       true      pred  count
3  Enrolled  Graduate     31
1   Dropout  Graduate     30
2  Enrolled   Dropout     24
0   Dropout  Enrolled     13
5  Graduate  Enrolled     11
4  Graduate   Dropout     10

=== Random Forest ===
Total errors: 209
Extreme errors (confidence > 0.7): 41
       true      pred  count
2  Enrolled  Graduate     19
0   Dropout  Graduate     12
1  Enrolled   Dropout      9
3  Graduate   Dropout      1


Random Forest - balanced 

In [62]:
sw = compute_sample_weight("balanced", y_train)

rf_balanced = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42)
rf_balanced.fit(X_train, y_train)

y_pred_rf = rf_balanced.predict(X_test)
y_proba_rf = rf_balanced.predict_proba(X_test)

results_rf = pd.DataFrame({
    "true": le.inverse_transform(y_test),
    "pred": le.inverse_transform(y_pred_rf),
    "confidence": y_proba_rf.max(axis=1),
})
results_rf["correct"] = results_rf["true"] == results_rf["pred"]
extreme_rf = results_rf[~results_rf["correct"] & (results_rf["confidence"] > 0.7)]

print(f"\n=== Random Forest (Balanced) ===")
print(f"Total errors: {(~results_rf['correct']).sum()}")
print(f"Extreme errors (confidence > 0.7): {len(extreme_rf)}")
print(extreme_rf.groupby(["true", "pred"]).size().reset_index(name="count").sort_values("count", ascending=False))


=== Random Forest (Balanced) ===
Total errors: 197
Extreme errors (confidence > 0.7): 41
       true      pred  count
2  Enrolled  Graduate     19
0   Dropout  Graduate     13
1  Enrolled   Dropout      9
